Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Master Course #24512

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# Principal Component Analysis (PCA) on Full-Column Rank Data Matrix 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from numpy.linalg import matrix_rank
from scipy.linalg import eigh, norm, svd, diagsvd

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(1)

PC...principal component

The term *principal component* is ambiguously used in the literature for
- either the feature signals in the new space
- or the eigen-/singular vectors
- or the feature signals in the new space with some scaling
- or the eigen-/singular vectors with some scaling.

We need thus to be careful to not mix up different of these PCA conventions.
Each convention itself is consistent, but it gets really messy when mixing them up or confusing the wording.
We might alternatively use *PC feature* / *PC signal* and *PC loading* / *PC eigenvector* to (hopefully) better clarify what we mean by the referred matrices.
The best way is to think in terms of the math and its essence, such that we actually don't need fancy words for what is going on.
But of course some very precise names are useful as well. 

Tall / Thin data matrix $\mathbf{X}$:

- we put features / signals into columns
- column index is along $1 \leq f \leq F$
- the data samples of a feature come along $1 \leq n \leq N$, that is the row index
- we assume $N \gg F$, hence the data matrix of shape $N \times F$ is tall / thin.
- we assume full column rank, i.e. rank of $\mathbf{X}$ is $F$.

Very often the PCA is discussed with the transposed data matrix $\mathbf{X}^T$ of shape $F \times N$,
i.e. a short / wide, full row rank data matrix is used where the features are in the rows.
We are careful, to not mix up different PCA approaches.

In [ ]:
# load a N=34 x F=5 real problem data matrix
X = np.loadtxt(open("pca_introduction.csv", "rb"),
               delimiter=";", skiprows=0)

In [ ]:
# or create a N=1000 x F=2 toy data matrix
if False:
    mean = [0, 0]
    cov = [[1, 0.85], [0.85, 1]]
    N = 1000
    x, y = rng.multivariate_normal(mean, cov, N).T
    X = np.array([x, y]).T
    X -= np.mean(X, axis=0)

In [ ]:
N, F = X.shape
N, F, F==matrix_rank(X)

In [ ]:
# zscore X and normalise by 1/sqrt(N-1) towards Z:
Z = np.copy(X)

# de-mean the columns is essential for PCA
# then the EVD and the SVD approaches for
# calculating the PCA yield the same results
Z -= np.mean(Z, axis=0)

if True:  # note: most plots are designed for True
    # these are optional, but very often done, steps for PCA
    # they altering the result's interpretation
    # if features have different physical units
    # std-normalisation is actually mandatory 
    Z /= np.std(Z, axis=0, ddof=1)  
    Z /= np.sqrt(N-1)

In [ ]:
# unbiased estimator for covariance matrix
CovZ = Z.T @ Z
CovZ  # with above normalisation we have
# a correlation matrix with unit diagonal

## PCA Essence

PCA is about
1. to find new features (PC signals, PC features),
such that their covariance matrix shows that all the
new features are completely independent, in other words:
PC features shall be orthogonal -> then its covariance matrix is diagonal
2. to sort the variances, such that the first new feature (PC feature 1) has
most variance, and the last new feature (PC feature F) the least variance. The (what we here call)
PC eigenvectors show into the orthogonal directions along the axis, these variances occur.

This is actually an optimisation problem.
Its result and essence can be motivated by
- the SVD of the data matrix $Z$
- the EVD of the covariance matrix $Z^\text{T} Z$

In [ ]:
# get the economy-sized SVD of Z
U, s, Vh = svd(Z, full_matrices=False)
S = diagsvd(s, F, F)
V = Vh.T
U.shape, S.shape, V.shape, np.std(U, axis=0, ddof=1) - 1/np.sqrt(N-1)

In [ ]:
# get sorted variances from squared singular values
variance = np.diag(S.T @ S)  # note: S.T==S by definition
variance

In [ ]:
np.cumsum(variance)  # cumsum variances

In [ ]:
np.sum(variance), F  # total variance == F

In [ ]:
plt.subplot(2,1,1)
plt.plot(np.arange(F)+1, variance, 'ok--')
plt.xticks(np.arange(F)+1)
plt.ylabel(r'variance = $\lambda$ of CovZ = $\sigma^2$ of Z')
plt.title('scree plot')
plt.grid(True)
plt.subplot(2,1,2)
plt.plot(np.arange(F)+1, np.cumsum(variance), 'ok--')
plt.xticks(np.arange(F)+1)
plt.ylim([0,F+0.2])
plt.xlabel('eigen-/singular value index')
plt.ylabel('cumsum variance')
plt.grid(True)
variance, np.cumsum(variance)

In [ ]:
# get the EVD of CovZ
[L, Q] = eigh(CovZ)
idx = np.flip(np.argsort(L))
L = L[idx]  # sort eigen values == variances
L = np.diag(L)
Q = Q[:, idx]  # and eigen vectors ->
# these are orthonormal because CovZ is pos definite

# check that SVD's V-vectors and eig's Q-vectors
# only differ in polarity
print(np.allclose(np.abs(V/Q), np.ones((F,F))))
# then we can match V and Q
# by inverting Q's polarities where needed:
Q = Q @ np.diag(np.diag(V/Q))
print(np.allclose(V,Q))
print('variances:')
variance, np.diag(L), np.diag(S.T @ S)

In the remainder we use only the SVD matrices for further calculations.

## Mindset: Orthonormal Eigenvectors

The PCA is essentially about the factorisation

$$\mathbf{Z} = \text{PC\_features} \cdot \text{PC\_loadings}^\text{T}$$

which can be derived from the economy-sized SVD

$$\mathbf{Z} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^\text{T}$$

From the above eigenwert problem for $\mathbf{Z}^\text{T} \mathbf{Z}$ we got $\mathbf{Q}$ and checked that these eigenvectors are orthonormal and consequently $\mathbf{Q}=\mathbf{V}$.

A PCA convention that uses these orthonormal eigenvectors, then has

$$\text{PC\_features} = \mathbf{U} \mathbf{\Sigma}$$

$$\text{PC\_loadings} = \mathbf{V}$$

It is straightforward to show that the covariance matrix of the $\text{PC\_features}$ is diagonal as desired

$$\text{PC\_features}^\text{T} \text{PC\_features} = (\mathbf{U} \mathbf{\Sigma})^\text{T} (\mathbf{U} \mathbf{\Sigma}) = \mathbf{\Sigma}^\text{T} \mathbf{\Sigma}$$

Due to sorted singular values we also immediately see that the variances along the diagonal are sorted as desired.


In [ ]:
# the PCA with the mindset of orthonormal eigenvectors in V=Q
# is given by the SVD as follows
PC_loadings = V
PC_features1 = U @ S  # option 1
PC_features2 = Z @ V  # option 2
np.allclose(PC_features1, PC_features2)
# simply because by SVD definition Z = U @ S @ V.T we have
# Z @ V = U @ S @ V.T @ V = U S

In [ ]:
# the eigenvetors have unit length in this mindset
norm(PC_loadings[:,0]), norm(PC_loadings[:,1])  # ...

In [ ]:
# the PCA is an unsupervised approach to diagonalise
# the covariance matrix of
# new features Y == PC_features == PC signals
#  
# we process Z features towards Y features by
# orthogonalizing with the SVD's V = EVD's Q
# orthonormal input basis vectors
# this yields
Y = U @ S  # PC_features1 ==
Y = Z @ V  # PC_features2

# then the covariance matrix
CovY = Y.T @ Y
# is indeed diagonal, i.e. these new features are orthogonal
CovY, np.sum(Y**2, axis=0), variance
# with largest variance top left and lowest variance bottom right
# matching the squared sums of the Y columns
# matching the squared singular values of Z
# matching the eigen values of Z'Z

In [ ]:
# check that matrix factorisation is consistent
np.allclose(Z, PC_features1 @ PC_loadings.T)

In [ ]:
# show data matrix in the original feature space
# and in the PC feature space
# this is convenient only in 2D (and 3D)
# we here go for 2D only, e.g. use the 2D toy data from above
if F==2:
    plt.figure(figsize=(10,8))
    plt.subplot(2,1,1)
    plt.plot(Z[:,0]*np.sqrt(N-1), Z[:,1]*np.sqrt(N-1), 'o', ms=4)
    plt.plot([-V[0,0], +V[0,0]], [-V[1,0], +V[1,0]], 'C3', label=r'largest data spread along $v_1$ axis')
    plt.plot([-V[0,1], +V[0,1]], [-V[1,1], +V[1,1]], 'C1', label=r'smallest data spread along $v_2$ axis')
    plt.axis('equal'), plt.xlim([-10, 10]), plt.ylim([-10,10])
    plt.legend()
    plt.xlabel(r'original feature dim 1: column 1 of $\sqrt{N-1} \cdot Z$ ')
    plt.ylabel(r'original feature dim 2: column 2 of $\sqrt{N-1} \cdot Z$ ')
    plt.grid()

    plt.subplot(2,1,2)
    plt.plot(Y[:,0]*np.sqrt(N-1), Y[:,1]*np.sqrt(N-1), 'o', ms=4)
    plt.plot([-1, +1], [0, 0], 'C3')
    plt.plot([0, 0], [-1, +1], 'C1')
    plt.axis('equal'), plt.xlim([-10, 10]), plt.ylim([-10,10])
    plt.xlabel(r'PC feature dim 1: column 1 of $\sqrt{N-1} \cdot Y$ ')
    plt.ylabel(r'PC feature dim 2: column 2 of $\sqrt{N-1} \cdot Y$ ')
    plt.grid()

    plt.tight_layout()

## Mindset: Orthonormal PC Features

Again: the PCA is essentially about the factorisation

$$\mathbf{Z} = \text{PC\_features} \cdot \text{PC\_loadings}^\text{T}$$

which can be derived from the economy-sized SVD

$$\mathbf{Z} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^\text{T}$$

From the above eigenwert problem for $\mathbf{Z}^\text{T} \mathbf{Z}$ we got $\mathbf{Q}$ and checked that these eigenvectors are orthonormal and consequently $\mathbf{Q}=\mathbf{V}$.


Now we rather might be interested to directly indicate the correlation between the columns in $\mathbf{Z}$ and ortho**normal** columns of $\text{PC\_features}$.

This is achieved by simply re-ordering the SVD's matrix factorisation to

$$\text{PC\_features} = \mathbf{U}$$

$$\text{PC\_loadings} = \mathbf{V} \mathbf{\Sigma}^\text{T}$$

Note that for economy-sized SVD we have $\mathbf{\Sigma}^\text{T} = \mathbf{\Sigma}$.

The columns of $\text{PC\_features}$ are by SVD's definition orthonormal, i.e. $\mathbf{U}^\text{T} \mathbf{U} = \mathbf{I}$

The resulting, often called loading matrix $\text{PC\_loadings}$ is indeed a correlation matrix as seen below. Or with another viewpoint:
$\text{PC\_loadings}^\text{T}$ contains the weights for linear combinations of $\text{PC\_features}$ columns.

We might now get an idea, why the term *principal component* could be misunderstood, if different literature refers to different
of these (and many more) possible matrix factorisations based on the SVD matrices.

In [ ]:
PC_features = U
# the covariance matrix of PC_features signals
# is diagonal, so we know
# that the PC_features columns are orthogonal
# due to unit diagonal, they are even orthonormal
# this tells us that columns in PC_features have unit variance
PC_features.T @ PC_features, np.sum(PC_features**2, axis=0)

In [ ]:
PC_loadings = V @ S.T  # loading matrix

In [ ]:
# check that matrix factorisation is consistent
np.allclose(Z, PC_features @ PC_loadings.T)

now the sorted variances are found in `PC_loadings`

In [ ]:
np.diag(PC_loadings.T @ PC_loadings), variance

In [ ]:
# we still deal with the same eigenwert problem for Z.T @ Z:
# eigen vectors = columns in PC_loadings
# eigen values = diagonal of L
np.allclose(CovZ @ PC_loadings, PC_loadings @ L)

In [ ]:
# but here the eigenvetors have no unit length
norm(PC_loadings[:,0]), norm(PC_loadings[:,1])  # ...

In [ ]:
PC_loadings
# this is a correlation matrix to be read as
# "input feature Z at row index i loads to PC feature at column index j"
# row index i:    input feature Z
# column index j: PC feature

In [ ]:
if F==5:
    feature_label = ['feature Z 1', 'feature Z 2', 'feature Z 3', 'feature Z 4', 'feature Z 5']
    pcf_label = ['PCF 1', 'PCF 2', 'PCF 3', 'PCF 4', 'PCF 5']
    cmap = plt.get_cmap('Spectral_r', 8)
    fig = plt.figure(figsize=(6,4))
    ax = fig.add_subplot(111)
    cax = ax.matshow(PC_loadings,
                     cmap=cmap,
                     vmin=-1, vmax=+1)
    clb = fig.colorbar(cax)
    ax.set_xticks(np.arange(len(pcf_label)))
    ax.set_yticks(np.arange(len(feature_label)))
    ax.set_xticklabels(pcf_label)
    ax.set_yticklabels(feature_label)
    clb.set_label('correlation', loc='center', labelpad=0)
    ax.set_title('loading matrix: feature Z i loads to PC feature j')
    plt.tight_layout()

The notebook [pca_audio_features.ipynb](pca_audio_features.ipynb) applies a PCA on audio features and presents some interpretation on the loading matrix. It shows that the 12 chosen audio features load highly to just three PC features. This tells us that the chosen audio features exhibit a certain correlation.

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.